In [ ]:
import faiss
import numpy as np

d = 384                      # embedding dim, e.g. all-MiniLM-L6-v2
xb = np.random.rand(100_000, d).astype("float32")  # your corpus embeddings

# --- exact baseline ---
index_flat = faiss.IndexFlatIP(d)     # inner product; use IndexFlatL2 for L2
index_flat.add(xb)

# --- HNSW: best default for RAG-scale corpora (10^4 - 10^7 vectors) ---
index_hnsw = faiss.IndexHNSWFlat(d, 32)     # M = 32 links per node
index_hnsw.hnsw.efConstruction = 200        # build-time search breadth
index_hnsw.add(xb)
index_hnsw.hnsw.efSearch = 64               # query-time search breadth (recall/latency knob)

# --- IVF-PQ: for 10^7+ vectors where RAM is the constraint ---
quantizer = faiss.IndexFlatIP(d)
index_ivfpq = faiss.IndexIVFPQ(quantizer, d, 4096, 8, 8)  # nlist=4096, m=8 subquantizers, 8 bits
index_ivfpq.train(xb)
index_ivfpq.add(xb)
index_ivfpq.nprobe = 16                     # cells searched per query

query = np.random.rand(1, d).astype("float32")
D, I = index_hnsw.search(query, k=5)        # returns distances, indices of top-5